# Anomaly Detection EDA — 23-KA-9101 Compressor

Compare z-score and Isolation Forest anomaly methods on 12 months of data.
Run all cells top-to-bottom after the pipeline has written `data/oid.duckdb`.

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

conn = duckdb.connect("../data/oid.duckdb", read_only=True)
print("Connected. Tables:", [r[0] for r in conn.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema='main' ORDER BY table_name"
).fetchall()])

## 1. Anomaly summary statistics

In [ ]:
# Overall counts
summary = conn.execute("""
    SELECT method,
           count(*) AS total_rows,
           sum(CASE WHEN is_anomaly THEN 1 ELSE 0 END) AS flagged
    FROM anomalies
    GROUP BY method
""").fetchdf()
display(summary)

# Z-score flags by signal
zs_by_signal = conn.execute("""
    SELECT signal, count(*) AS n_flags
    FROM anomalies
    WHERE method = 'zscore' AND is_anomaly
    GROUP BY signal ORDER BY n_flags DESC
""").fetchdf()
display(zs_by_signal)

## 2. Daily anomaly counts vs. running hours

In [ ]:
kpis = conn.execute("SELECT * FROM v_daily_kpis").fetchdf()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=kpis["day"], y=kpis["anomaly_count"], name="Anomaly count",
           marker_color="#e45756", opacity=0.7),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=kpis["day"], y=kpis["running_hours"], name="Running hours",
               line=dict(color="#4c78a8", width=1.5)),
    secondary_y=True,
)
fig.update_layout(title="Daily anomaly count vs running hours", height=400,
                  legend=dict(orientation="h", y=1.12))
fig.update_yaxes(title_text="Anomaly count", secondary_y=False)
fig.update_yaxes(title_text="Running hours", secondary_y=True)
fig.show()

## 3. Z-score flags by signal over time

Each dot is a z-score flag; y-axis shows |z|. Bearing temperatures tend to
have more flags than vibration since their rolling stats have lower variance.

In [ ]:
zs = conn.execute("""
    SELECT ts, signal, score
    FROM anomalies
    WHERE method = 'zscore' AND is_anomaly
""").fetchdf()

fig = px.scatter(zs, x="ts", y="score", color="signal",
                 title="Z-score anomalies over time",
                 labels={"ts": "Date", "score": "|z|"},
                 opacity=0.6, height=450)
fig.update_layout(legend=dict(orientation="h", y=-0.2))
fig.show()

## 4. Isolation Forest scores over time

Score = −decision_function (higher = more anomalous). Red dots are flagged.
Grey shaded regions mark offline (shutdown) periods.

In [ ]:
ifo = conn.execute("""
    SELECT ts, score, is_anomaly
    FROM anomalies
    WHERE method = 'iforest'
    ORDER BY ts
""").fetchdf()

# Compute contiguous offline windows for shading
offline_windows = conn.execute("""
    WITH numbered AS (
        SELECT ts,
               ts - INTERVAL 10 MINUTE * row_number() OVER (ORDER BY ts) AS grp
        FROM features_10m
        WHERE NOT is_running
    )
    SELECT min(ts) AS start_ts, max(ts) + INTERVAL 10 MINUTE AS end_ts
    FROM numbered
    GROUP BY grp
    ORDER BY start_ts
""").fetchdf()

fig = go.Figure()
normal = ifo[~ifo["is_anomaly"]]
anomalous = ifo[ifo["is_anomaly"]]
fig.add_trace(go.Scatter(x=normal["ts"], y=normal["score"], mode="markers",
                         marker=dict(size=3, color="#4c78a8", opacity=0.3),
                         name="Normal"))
fig.add_trace(go.Scatter(x=anomalous["ts"], y=anomalous["score"], mode="markers",
                         marker=dict(size=5, color="#e45756"),
                         name="Anomaly"))

for _, w in offline_windows.iterrows():
    fig.add_vrect(
        x0=w["start_ts"], x1=w["end_ts"],
        fillcolor="grey", opacity=0.25, line_width=0,
        annotation_text="offline" if (w["end_ts"] - w["start_ts"]).total_seconds() > 7200 else "",
        annotation_position="top left",
        annotation_font_size=9,
        annotation_font_color="grey",
    )

fig.update_layout(title="Isolation Forest scores over time (grey = offline periods)",
                  xaxis_title="Date", yaxis_title="Score (−decision_function)",
                  height=450)
fig.show()

## 5. Anomaly windows

Consecutive flagged rows merged when gap ≤ 30 min.

In [ ]:
windows = conn.execute("""
    SELECT window_id, method, signal, start_ts, end_ts, max_score, n_points
    FROM v_anomaly_windows
    ORDER BY max_score DESC
""").fetchdf()

print(f"Total anomaly windows: {len(windows)}")
print(f"  Z-score windows: {len(windows[windows.method == 'zscore'])}")
print(f"  IForest windows: {len(windows[windows.method == 'iforest'])}")
print()
print("Top 10 windows by max_score:")
display(windows.head(10))

## 6. Method agreement — do z-score and iforest flag the same time periods?

For each iforest anomaly timestamp, check whether z-score also flagged
that timestamp (any signal).

In [ ]:
overlap = conn.execute("""
    WITH ifo_flags AS (
        SELECT DISTINCT ts FROM anomalies WHERE method = 'iforest' AND is_anomaly
    ),
    zs_flags AS (
        SELECT DISTINCT ts FROM anomalies WHERE method = 'zscore' AND is_anomaly
    )
    SELECT
        (SELECT count(*) FROM ifo_flags) AS iforest_only_ts,
        (SELECT count(*) FROM ifo_flags WHERE ts IN (SELECT ts FROM zs_flags)) AS both_methods,
        (SELECT count(*) FROM zs_flags) AS zscore_only_ts
""").fetchdf()
display(overlap)

## 7. Drill-down: worst anomaly day

Pick the day with the most anomaly flags and plot the raw signals
alongside anomaly markers.

In [ ]:
worst_day = kpis.sort_values("anomaly_count", ascending=False).iloc[0]["day"]
print(f"Worst anomaly day: {worst_day}")

day_features = conn.execute(f"""
    SELECT ts, vib_mtr_de_1, vib_mtr_de_2, t_bearing_comp_de, t_bearing_comp_nde,
           pressure_ratio, is_running
    FROM features_10m
    WHERE CAST(ts AS DATE) = '{worst_day}'
    ORDER BY ts
""").fetchdf()

day_anomalies = conn.execute(f"""
    SELECT ts, signal, method, score
    FROM anomalies
    WHERE is_anomaly AND CAST(ts AS DATE) = '{worst_day}'
""").fetchdf()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=["Vibration (DE)", "Bearing temps", "Pressure ratio"])

fig.add_trace(go.Scatter(x=day_features["ts"], y=day_features["vib_mtr_de_1"],
                         name="vib_de_1", line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=day_features["ts"], y=day_features["vib_mtr_de_2"],
                         name="vib_de_2", line=dict(width=1)), row=1, col=1)

fig.add_trace(go.Scatter(x=day_features["ts"], y=day_features["t_bearing_comp_de"],
                         name="t_brg_comp_de", line=dict(width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=day_features["ts"], y=day_features["t_bearing_comp_nde"],
                         name="t_brg_comp_nde", line=dict(width=1)), row=2, col=1)

fig.add_trace(go.Scatter(x=day_features["ts"], y=day_features["pressure_ratio"],
                         name="pressure_ratio", line=dict(width=1)), row=3, col=1)

# Overlay anomaly markers
for method, color in [("zscore", "#e45756"), ("iforest", "#f58518")]:
    subset = day_anomalies[day_anomalies["method"] == method]
    if len(subset) > 0:
        fig.add_trace(go.Scatter(x=subset["ts"], y=[0]*len(subset), mode="markers",
                                 marker=dict(symbol="triangle-up", size=8, color=color),
                                 name=f"{method} flag", showlegend=True), row=3, col=1)

fig.update_layout(height=700, title=f"Drill-down: {worst_day}",
                  legend=dict(orientation="h", y=-0.05))
fig.show()

## 8. Anomalies near events

If the events table is populated, this shows anomaly windows that occurred
within ±7 days of a maintenance or process event.

In [ ]:
near = conn.execute("SELECT * FROM v_anomalies_near_events").fetchdf()
n_events = conn.execute("SELECT count(*) FROM events").fetchone()[0]

if n_events == 0:
    print("No events in the database — this view will be useful once events are extracted.")
else:
    print(f"{len(near)} anomaly-event pairs found:")
    display(near.head(20))

## 9. Model evaluation — can anomaly flags predict shutdowns?

For each offline window (≥1h duration), check whether anomaly flags appeared
in the preceding 1–7 days. A flag *after* the shutdown is hindsight, not prediction.

**Metrics:**
- **Recall:** fraction of shutdowns preceded by at least one anomaly flag
- **Precision:** fraction of anomaly windows that precede a shutdown (within 7 days)
- **Lead time:** how far in advance the first flag appeared before each shutdown

In [ ]:
# Offline windows ≥1h as "shutdown events" to predict
shutdowns = conn.execute("""
    WITH numbered AS (
        SELECT ts,
               ts - INTERVAL 10 MINUTE * row_number() OVER (ORDER BY ts) AS grp
        FROM features_10m
        WHERE NOT is_running
    )
    SELECT min(ts) AS start_ts, max(ts) AS end_ts, count(*) AS n_points,
           EXTRACT(EPOCH FROM max(ts) - min(ts)) / 3600 AS duration_hrs
    FROM numbered
    GROUP BY grp
    HAVING count(*) >= 6
    ORDER BY start_ts
""").fetchdf()

print(f"Shutdown events (offline ≥1h): {len(shutdowns)}")
display(shutdowns)

# Anomaly windows by method
anom_windows = conn.execute("""
    SELECT window_id, method, signal, start_ts, end_ts, max_score, n_points
    FROM v_anomaly_windows
    ORDER BY start_ts
""").fetchdf()

LOOKAHEAD_DAYS = 7

results = []
for method in ["zscore", "iforest"]:
    mw = anom_windows[anom_windows["method"] == method]
    total_anom_windows = len(mw)

    shutdowns_with_warning = 0
    lead_times = []
    anom_windows_before_shutdown = set()

    for _, sd in shutdowns.iterrows():
        sd_start = sd["start_ts"]
        window_start = sd_start - pd.Timedelta(days=LOOKAHEAD_DAYS)
        preceding = mw[(mw["end_ts"] >= window_start) & (mw["end_ts"] < sd_start)]

        if len(preceding) > 0:
            shutdowns_with_warning += 1
            first_flag = preceding["start_ts"].min()
            lead_hrs = (sd_start - first_flag).total_seconds() / 3600
            lead_times.append(lead_hrs)
            anom_windows_before_shutdown.update(preceding["window_id"].values)

    recall = shutdowns_with_warning / len(shutdowns) if len(shutdowns) > 0 else 0
    precision = len(anom_windows_before_shutdown) / total_anom_windows if total_anom_windows > 0 else 0

    results.append({
        "method": method,
        "shutdown_events": len(shutdowns),
        "shutdowns_with_prior_flag": shutdowns_with_warning,
        "recall": f"{recall:.0%}",
        "anomaly_windows_total": total_anom_windows,
        "anomaly_windows_before_shutdown": len(anom_windows_before_shutdown),
        "precision": f"{precision:.0%}",
        "median_lead_time_hrs": f"{pd.Series(lead_times).median():.1f}" if lead_times else "N/A",
        "min_lead_time_hrs": f"{min(lead_times):.1f}" if lead_times else "N/A",
        "max_lead_time_hrs": f"{max(lead_times):.1f}" if lead_times else "N/A",
    })

eval_df = pd.DataFrame(results)
print(f"\nPredictive evaluation (lookahead = {LOOKAHEAD_DAYS} days):")
display(eval_df)

In [ ]:
# Timeline: anomaly flags leading into shutdown events
fig = go.Figure()

# Anomaly windows as horizontal bars
for method, color, ybase in [("zscore", "#e45756", 0), ("iforest", "#f58518", 1)]:
    mw = anom_windows[anom_windows["method"] == method]
    for _, w in mw.iterrows():
        fig.add_trace(go.Scatter(
            x=[w["start_ts"], w["end_ts"]], y=[ybase, ybase],
            mode="lines", line=dict(color=color, width=4),
            showlegend=False, hovertext=f"{method} | score={w['max_score']:.2f}",
            hoverinfo="text+x",
        ))

# Add one legend entry per method
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color="#e45756", width=4), name="Z-score anomaly"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color="#f58518", width=4), name="IForest anomaly"))

# Shutdown periods as grey bands
for _, sd in shutdowns.iterrows():
    fig.add_vrect(
        x0=sd["start_ts"], x1=sd["end_ts"],
        fillcolor="grey", opacity=0.3, line_width=0,
    )
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
    marker=dict(size=12, color="grey", opacity=0.3, symbol="square"),
    name="Shutdown period"))

fig.update_layout(
    title="Anomaly windows vs shutdown periods (grey bands)",
    xaxis_title="Date", yaxis=dict(
        tickvals=[0, 1], ticktext=["Z-score", "IForest"],
        range=[-0.5, 1.8],
    ),
    height=300, showlegend=True,
    legend=dict(orientation="h", y=1.15),
)
fig.show()

## 10. Drill-down: signals around each shutdown

For each significant offline period, plot the raw signals ±3 days around the
shutdown start, with anomaly markers overlaid. This reveals whether the
anomaly detectors flagged anything *before* the machine went offline.

In [ ]:
CONTEXT_DAYS = 3

for idx, sd in shutdowns.iterrows():
    window_start = sd["start_ts"] - pd.Timedelta(days=CONTEXT_DAYS)
    window_end = sd["end_ts"] + pd.Timedelta(days=CONTEXT_DAYS)

    feat = conn.execute(f"""
        SELECT ts, vib_mtr_de_1, vib_mtr_de_2, vib_mtr_nde_1, vib_mtr_nde_2,
               t_bearing_comp_de, t_bearing_comp_nde, t_bearing_mtr_de, t_bearing_mtr_nde,
               pressure_ratio, is_running
        FROM features_10m
        WHERE ts BETWEEN '{window_start}' AND '{window_end}'
        ORDER BY ts
    """).fetchdf()

    anoms = conn.execute(f"""
        SELECT ts, signal, method, score
        FROM anomalies
        WHERE is_anomaly
          AND ts BETWEEN '{window_start}' AND '{window_end}'
    """).fetchdf()

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=["Vibration", "Bearing temps", "Pressure ratio + flags"])

    # Vibration
    for sig in ["vib_mtr_de_1", "vib_mtr_de_2", "vib_mtr_nde_1", "vib_mtr_nde_2"]:
        if sig in feat.columns:
            fig.add_trace(go.Scatter(x=feat["ts"], y=feat[sig], name=sig,
                                     line=dict(width=1)), row=1, col=1)

    # Bearing temps
    for sig in ["t_bearing_comp_de", "t_bearing_comp_nde", "t_bearing_mtr_de", "t_bearing_mtr_nde"]:
        if sig in feat.columns:
            fig.add_trace(go.Scatter(x=feat["ts"], y=feat[sig], name=sig,
                                     line=dict(width=1)), row=2, col=1)

    # Pressure ratio
    fig.add_trace(go.Scatter(x=feat["ts"], y=feat["pressure_ratio"],
                             name="pressure_ratio", line=dict(width=1, color="#4c78a8")),
                  row=3, col=1)

    # Anomaly markers on pressure ratio subplot
    for method, color, symbol in [("zscore", "#e45756", "triangle-up"),
                                   ("iforest", "#f58518", "diamond")]:
        subset = anoms[anoms["method"] == method]
        if len(subset) > 0:
            fig.add_trace(go.Scatter(
                x=subset["ts"], y=[feat["pressure_ratio"].min()] * len(subset),
                mode="markers",
                marker=dict(symbol=symbol, size=8, color=color),
                name=f"{method} flag",
            ), row=3, col=1)

    # Shade the shutdown period
    fig.add_vrect(x0=sd["start_ts"], x1=sd["end_ts"],
                  fillcolor="grey", opacity=0.25, line_width=0)

    n_zs = len(anoms[anoms["method"] == "zscore"])
    n_if = len(anoms[anoms["method"] == "iforest"])
    fig.update_layout(
        height=700,
        title=f"Shutdown {idx+1}: {sd['start_ts'].strftime('%Y-%m-%d %H:%M')} "
              f"({sd['duration_hrs']:.0f}h offline) — {n_zs} zscore / {n_if} iforest flags in ±{CONTEXT_DAYS}d",
        legend=dict(orientation="h", y=-0.05),
    )
    fig.show()

In [ ]:
conn.close()
print("Done.")